# eXplainable AI (SHAP) for Neural Network
We use `shap.KernelExplainer` to interpret the MLP model predictions since NNs are black-box models.


In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
shap.initjs()


In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
if 'id' in df.columns: df.drop('id', axis=1, inplace=True)
df = df[df['gender'] != 'Other']
df_impute = df.copy()
for col in df_impute.select_dtypes(include=['object', 'string']).columns:
    df_impute[col] = LabelEncoder().fit_transform(df_impute[col].astype(str))
df['bmi'] = KNNImputer(n_neighbors=2).fit_transform(df_impute)[:, df_impute.columns.get_loc('bmi')].round(1)
df = df[(df['bmi'] >= df['bmi'].quantile(0.001)) & (df['bmi'] <= df['bmi'].quantile(0.999))]
df['work_type'] = df['work_type'].map({'Govt_job': 4, 'Never_worked': 1, 'Private': 3, 'Self-employed': 2, 'children': 0})
df['smoking_status'] = df['smoking_status'].map({'Unknown': 1, 'formerly smoked': 2, 'never smoked': 0, 'smokes': 3})
for col in ['gender', 'ever_married', 'Residence_type']: df[col] = LabelEncoder().fit_transform(df[col])
X = df.drop('stroke', axis=1)
y = df['stroke']
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

smote = SMOTE(random_state=59)
X_res, y_res = smote.fit_resample(X_scaled, y)
mlp = MLPClassifier(hidden_layer_sizes=(128, 64, 32), solver='lbfgs', alpha=0.01, random_state=59, max_iter=500)
mlp.fit(X_res, y_res)


## Explaining Global Feature Importance with SHAP


In [ ]:
# KernelExplainer can be slow, so we use a k-means background summary of the training data
background = shap.kmeans(X_scaled, 50)
explainer = shap.KernelExplainer(mlp.predict_proba, background)

# Calculate SHAP values for a subset to save time
shap_values = explainer.shap_values(X_scaled.sample(200, random_state=59))

# We take the SHAP values for the positive class (Stroke = 1)
if isinstance(shap_values, list):
    shap_vals_positive = shap_values[1]
else:
    # In some shap versions, shape is (n_samples, n_features, n_classes)
    if len(shap_values.shape) == 3:
        shap_vals_positive = shap_values[:, :, 1]
    else:
        shap_vals_positive = shap_values


In [ ]:
shap.summary_plot(shap_vals_positive, X_scaled.sample(200, random_state=59), feature_names=X.columns)


## Local Explanation for a Single Patient


In [ ]:
sample_idx = 0
shap.force_plot(explainer.expected_value[1], shap_vals_positive[sample_idx,:], X_scaled.iloc[sample_idx,:], feature_names=X.columns)
